# Compare small response models
Compare longer training, PCA focused on option differences, and a small nonlinear scorer. Keep eight respondent coordinates. Fit shared weights on background people and development vectors on anchors only. The eight validation questions select the model; final-test answers stay unread.


In [1]:
import hashlib
import json
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.decomposition import PCA
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir())
OUT = ROOT / "outputs"
def read_json(p): return json.loads((ROOT / p).read_text())
def sha256(p):
    with p.open("rb") as f: return hashlib.file_digest(f, "sha256").hexdigest()
config = read_json("configs/model_comparison.json")
split = read_json("data/splits/question_split.json")
people = read_json("data/splits/respondent_split.json")["groups"]
bank = read_json("data/processed/question_bank.json")
items = {i["item_id"]: i for i in bank["items"]}
index = pd.DataFrame(read_json("outputs/question_answer_index.json"))
embeddings = np.load(OUT / "question_answer_embeddings.npy", allow_pickle=False)
original_audit = read_json("outputs/shared_response_training.json")
for p, spec in original_audit["files"].items(): assert sha256(ROOT / p) == spec["sha256"]
embed_audit = read_json("outputs/question_answer_embedding_audit.json")
for p, spec in embed_audit["files"].items(): assert sha256(ROOT / p) == spec["sha256"]
input_names = ["configs/model_comparison.json", "data/splits/question_split.json", "data/splits/respondent_split.json",
               "data/processed/question_bank.json", "outputs/question_answer_embeddings.npy", "outputs/question_answer_index.json"]
input_names += list(original_audit["files"])
inputs = {p: sha256(ROOT / p) for p in input_names}

manifest = {p["partition"]: p for p in read_json("data/splits/partition_manifest.json")["partitions"]}
def load_partition(name, group, questions):
    spec = manifest[name]
    assert sha256(ROOT / spec["path"]) == spec["sha256"]
    inputs[spec["path"]] = spec["sha256"]
    frame = pd.read_parquet(ROOT / spec["path"])
    assert frame["pid"].tolist() == people[group]
    assert frame.columns.tolist() == ["pid"] + questions
    return frame

train_q = split["training_columns"]
anchor_q = [q for g in split["anchor_groups"].values() for q in g]
policy_q = split["validation_policy"]["columns"]
scale_q = split["validation_maximization"]["columns"]
val_q = policy_q + scale_q
assert not set(train_q) & set(split["forbidden_input_columns_all_respondents_all_waves"])
train = load_partition("shared_training", "shared_training", train_q)
anchors = load_partition("development_anchors", "development", anchor_q)
# This is the first model-selection use of validation labels. Final-test files are never loaded.
policy = load_partition("validation_policy", "development", policy_q)
scale = load_partition("validation_maximization", "development", scale_q)
validation = policy.merge(scale, on="pid", validate="one_to_one")
assert not set(train.pid) & set(anchors.pid)
assert not (set(train.pid) | set(anchors.pid)) & set(people["final_personas"])

def targets(frame, questions):
    a = frame[questions].to_numpy(dtype=float, na_value=np.nan)
    return np.where(np.isnan(a), -1, a - 1).astype(np.int64)
Y_train, Y_anchor, Y_val = targets(train, train_q), targets(anchors, anchor_q), targets(validation, val_q)
assert (Y_train < 0).sum() == 113 and (Y_anchor < 0).sum() == 24 and (Y_val >= 0).all()
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}; {len(config['candidates'])} fixed candidates; final test sealed.")


Device: mps; 7 fixed candidates; final test sealed.


## 1. Features and baselines
Original features remain unchanged. The alternative centres each question's option embeddings before fitting 128-component whitened PCA on training options only. This allocates PCA capacity to differences that affect answer choice.


In [2]:
training_rows = np.flatnonzero(index.item_id.isin(train_q).to_numpy())
assert len(training_rows) == 1822 and index.iloc[training_rows].question_role.eq("training").all()
centered = embeddings.copy()
for _, rows in index.groupby("item_id", sort=False):
    positions = rows.embedding_row.to_numpy()
    centered[positions] -= embeddings[positions].mean(axis=0)
pca = PCA(n_components=128, whiten=True, svd_solver="randomized", random_state=config["seed"])
pca.fit(centered[training_rows])
contrast = pca.transform(centered).astype(np.float32)
assert pca.n_samples_ == 1822 and np.isfinite(contrast).all()
feature_sets = {"original_64": np.load(OUT / "question_answer_features_64.npy", allow_pickle=False), "contrast_128": contrast}
contrast_path = OUT / "question_answer_features_contrast_128.npy"
contrast_pca_path = OUT / "question_answer_pca_contrast_128.npz"
np.save(contrast_path, contrast, allow_pickle=False)
np.savez(contrast_pca_path, mean=pca.mean_, components=pca.components_, explained_variance=pca.explained_variance_,
         explained_variance_ratio=pca.explained_variance_ratio_, training_embedding_rows=training_rows,
         center_each_question=np.array(True), whiten=np.array(True))
max_options = max(len(i["options"]) for i in bank["items"])

def question_tensors(features, questions):
    E = np.zeros((len(questions), max_options, features.shape[1]), np.float32)
    mask = np.zeros((len(questions), max_options), bool)
    for j, q in enumerate(questions):
        rows = index.loc[index.item_id.eq(q)].sort_values("answer_index")
        assert rows.answer_index.tolist() == list(range(len(rows)))
        E[j, :len(rows)] = features[rows.embedding_row.to_numpy()]
        mask[j, :len(rows)] = True
    return torch.tensor(E, device=device), torch.tensor(mask, device=device)

def metrics(probabilities, y, questions):
    observed = y >= 0
    safe_y = np.maximum(y, 0)
    chosen = np.take_along_axis(probabilities, safe_y[..., None], axis=-1)[..., 0]
    losses = -np.log(np.clip(chosen, 1e-12, 1))
    correct = probabilities.argmax(-1) == y
    per_question = []
    for j, q in enumerate(questions):
        valid = observed[:, j]; n = len(items[q]["options"])
        truth_share = np.bincount(y[valid, j], minlength=n) / valid.sum()
        predicted_share = probabilities[valid, j, :n].mean(axis=0)
        per_question.append({"item_id": q, "nll": float(losses[valid, j].mean()), "accuracy": float(correct[valid, j].mean()),
                             "distribution_TV": float(np.abs(truth_share - predicted_share).sum() / 2)})
    return {"nll": float(np.mean([p["nll"] for p in per_question])),
            "accuracy": float(correct[observed].mean()),
            "distribution_TV": float(np.mean([p["distribution_TV"] for p in per_question])), "per_question": per_question}

# Population baseline: pool training answers with identical option wording/order.
# It has no access to validation frequencies and gives everyone the same probabilities.
def signature(q): return tuple(o["label"] for o in items[q]["options"])
pooled = {}
modal_counts = []
for j, q in enumerate(train_q):
    n = len(items[q]["options"]); valid = Y_train[:, j] >= 0
    counts = np.bincount(Y_train[valid, j], minlength=n)
    key = signature(q)
    pooled[key] = pooled.get(key, np.zeros(n)) + counts
    modal_counts.append({"item_id": q, "block": items[q]["block"], "correct": int(counts.max()), "observed": int(valid.sum())})
population_probs = np.zeros((len(anchors), len(val_q), max_options), np.float32)
for j, q in enumerate(val_q):
    assert signature(q) in pooled
    counts = pooled[signature(q)] + 1
    population_probs[:, j, :len(counts)] = counts / counts.sum()
population_baseline = metrics(population_probs, Y_val, val_q)
training_modal_accuracy = sum(p["correct"] for p in modal_counts) / sum(p["observed"] for p in modal_counts)
print(f"Training question-majority accuracy: {training_modal_accuracy:.1%} (training-fit diagnostic only)")
print(f"Contrast PCA retains {pca.explained_variance_ratio_.sum():.1%} of training option-difference variance.")
# Second round: use training-only scale frequencies as a fixed population prior.
prior_column = []
for row in index.itertuples():
    key = signature(row.item_id)
    counts = pooled.get(key, np.zeros(len(key))) + 1
    prior_column.append(np.log(counts[row.answer_index] / counts.sum()))
feature_sets["prior_65"] = np.column_stack([feature_sets["original_64"], prior_column]).astype(np.float32)
prior_feature_path = OUT / "question_answer_features_prior_65.npy"
np.save(prior_feature_path, feature_sets["prior_65"], allow_pickle=False)
prior_path = OUT / "response_scale_priors.json"
prior_path.write_text(json.dumps([{"option_labels": list(key), "training_counts": counts.tolist(),
                                  "probabilities": ((counts + 1) / (counts.sum() + len(counts))).tolist()}
                                 for key, counts in pooled.items()], indent=2) + "\n")


Training question-majority accuracy: 52.7% (training-fit diagnostic only)
Contrast PCA retains 97.3% of training option-difference variance.


47436

## 2. Shared scorers
The linear model is unchanged. The nonlinear alternative maps answer features through a 64-unit tanh layer into one population score and eight preference features. No question-specific free parameters are introduced; both models can score unseen question text.


In [3]:
class ResponseModel(nn.Module):
    def __init__(self, dimension, architecture):
        super().__init__()
        self.architecture = architecture
        if architecture in {"linear", "prior_linear"}:
            dimension -= int(architecture == "prior_linear")
            self.beta = nn.Parameter(torch.zeros(dimension))
            self.W = nn.Parameter(torch.randn(8, dimension) * 0.05)
        else:
            self.head = nn.Sequential(nn.Linear(dimension, 64), nn.Tanh(), nn.Linear(64, 9))
            nn.init.normal_(self.head[-1].weight, std=0.05)
            nn.init.zeros_(self.head[-1].bias)
        self.respondents = nn.Embedding(len(train), 8)
        nn.init.normal_(self.respondents.weight, std=0.1)

    def shared(self, E):
        if self.architecture == "linear": return E @ self.beta, E @ self.W.T
        if self.architecture == "prior_linear": return E[..., -1] + E[..., :-1] @ self.beta, E[..., :-1] @ self.W.T
        output = self.head(E)
        return output[..., 0], output[..., 1:]

    def score(self, z, E, mask):
        base, preference = self.shared(E)
        return (base + torch.einsum("bd,qkd->bqk", z, preference)).masked_fill(~mask[None], -1e9)

    def shared_parameters(self):
        return [p for name, p in self.named_parameters() if name != "respondents.weight"]

loss_fn = nn.CrossEntropyLoss(ignore_index=-1)
train_y = torch.tensor(Y_train, device=device)
anchor_y = torch.tensor(Y_anchor, device=device)

def fit_shared(model, E, mask, epochs):
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
    rng = np.random.default_rng(config["seed"])
    history = []
    model.train()
    for epoch in range(1, epochs + 1):
        order = rng.permutation(len(train))
        for offset in range(0, len(order), config["batch_size"]):
            ids = torch.tensor(order[offset:offset + config["batch_size"]], device=device)
            z = model.respondents(ids)
            logits = model.score(z, E, mask)
            nll = loss_fn(logits.reshape(-1, max_options), train_y[ids].reshape(-1))
            penalty = sum(p.square().mean() for p in model.shared_parameters()) + z.square().mean()
            loss = nll + model.regularization * penalty
            assert torch.isfinite(loss).item()
            optimizer.zero_grad(set_to_none=True); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0, error_if_nonfinite=True)
            optimizer.step()
        if epoch % 100 == 0:
            history.append({"epoch": epoch, "last_batch_nll": float(nll.item())})
    model.eval()
    return history

@torch.no_grad()
def predict(model, z, E, mask):
    chunks = []
    for offset in range(0, len(z), config["batch_size"]):
        p = model.score(z[offset:offset + config["batch_size"]], E, mask).softmax(-1)
        assert torch.isfinite(p).all().item()
        torch.testing.assert_close(p.sum(-1), torch.ones_like(p[..., 0]), atol=1e-6, rtol=1e-6)
        assert p.masked_select(~mask[None].expand_as(p)).eq(0).all().item()
        chunks.append(p.cpu().numpy())
    return np.concatenate(chunks)

def infer_development(model, E, mask):
    model.requires_grad_(False)
    before = {name: p.detach().cpu().numpy().copy() for name, p in model.named_parameters()}
    z = nn.Parameter(model.respondents.weight.detach().mean(0).repeat(len(anchors), 1))
    optimizer = torch.optim.Adam([z], lr=config["development_learning_rate"])
    # Shared features are fixed; cache them during persona fitting.
    with torch.no_grad(): base, preference = model.shared(E)
    losses = []
    for step in range(config["development_steps"]):
        logits = (base + torch.einsum("bd,qkd->bqk", z, preference)).masked_fill(~mask[None], -1e9)
        nll = loss_fn(logits.reshape(-1, max_options), anchor_y.reshape(-1))
        loss = nll + model.regularization * z.square().mean()
        optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
        losses.append(float(nll.item()))
    assert losses[-1] < losses[0] and np.isfinite(losses).all()
    assert all(np.array_equal(before[name], p.detach().cpu().numpy()) for name, p in model.named_parameters())
    return z.detach(), {"initial_anchor_nll": losses[0], "final_anchor_nll": losses[-1]}


## 3. Fixed comparison and validation selection
The first five settings were specified before observing validation scores. Two prior-informed candidates were added after that first round; the config records why. Lower validation log loss wins; report accuracy and group-distribution error alongside it. Validation results are model-selection results, not an independent final assessment.


In [4]:
results, states = [], {}
for candidate in config["candidates"]:
    torch.manual_seed(config["seed"])
    features = feature_sets[candidate["features"]]
    train_E, train_mask = question_tensors(features, train_q)
    anchor_E, anchor_mask = question_tensors(features, anchor_q)
    val_E, val_mask = question_tensors(features, val_q)
    model = ResponseModel(features.shape[1], candidate["architecture"]).to(device)
    model.regularization = candidate.get("l2", config["l2_penalty"])
    if candidate.get("load_original"):
        with np.load(OUT / "shared_response_model.npz", allow_pickle=False) as saved:
            assert saved["person_ids"].tolist() == people["shared_training"]
            model.load_state_dict({"beta": torch.tensor(saved["beta"], device=device), "W": torch.tensor(saved["W"], device=device),
                                   "respondents.weight": torch.tensor(saved["respondent_vectors"], device=device)})
    start = time.monotonic()
    history = fit_shared(model, train_E, train_mask, candidate["epochs"])
    training_metrics = metrics(predict(model, model.respondents.weight.detach(), train_E, train_mask), Y_train, train_q)
    z, anchor_fit = infer_development(model, anchor_E, anchor_mask)
    probabilities = predict(model, z, val_E, val_mask)
    validation_metrics = metrics(probabilities, Y_val, val_q)
    policy_metrics = metrics(probabilities[:, :len(policy_q)], Y_val[:, :len(policy_q)], policy_q)
    if candidate.get("load_original"):
        assert abs(training_metrics["accuracy"] - original_audit["training"]["final"]["accuracy"]) < 1e-8
    result = {"name": candidate["name"], "spec": candidate, "parameters": sum(p.numel() for p in model.parameters()),
              "seconds": time.monotonic() - start, "training": training_metrics, "validation": validation_metrics,
              "policy_validation": policy_metrics, "anchor_fit": anchor_fit, "history": history}
    results.append(result)
    states[candidate["name"]] = {"weights": {name: p.detach().cpu().numpy().copy() for name, p in model.state_dict().items()},
                                  "development_vectors": z.cpu().numpy(), "probabilities": probabilities}
    print(f"{candidate['name']}: train accuracy {training_metrics['accuracy']:.1%}; validation accuracy {validation_metrics['accuracy']:.1%}, NLL {validation_metrics['nll']:.4f}; {result['seconds']:.1f}s", flush=True)

selected = min(results, key=lambda r: (r["validation"]["nll"], -r["validation"]["accuracy"]))
summary = pd.DataFrame([{"model": r["name"], "training_accuracy": r["training"]["accuracy"],
                         "validation_accuracy": r["validation"]["accuracy"], "validation_nll": r["validation"]["nll"],
                         "policy_validation_accuracy": r["policy_validation"]["accuracy"],
                         "validation_distribution_TV": r["validation"]["distribution_TV"]} for r in results])
display(summary)
print(f"Selected: {selected['name']} (lowest validation NLL)")
print(f"Pooled-scale population baseline: accuracy {population_baseline['accuracy']:.1%}, NLL {population_baseline['nll']:.4f}")


original_linear_64: train accuracy 51.6%; validation accuracy 34.3%, NLL 1.5761; 1.5s


longer_linear_64: train accuracy 51.6%; validation accuracy 34.5%, NLL 1.5748; 20.7s


contrast_linear_128: train accuracy 53.7%; validation accuracy 31.0%, NLL 1.6749; 19.7s


nonlinear_64: train accuracy 63.4%; validation accuracy 25.4%, NLL 2.9745; 24.3s


nonlinear_contrast_128: train accuracy 63.6%; validation accuracy 29.1%, NLL 2.5431; 26.9s


prior_linear_l2_003: train accuracy 55.9%; validation accuracy 35.5%, NLL 1.5392; 10.9s


prior_linear_l2_01: train accuracy 55.4%; validation accuracy 36.0%, NLL 1.5155; 11.1s


,model,training_accuracy,validation_accuracy,validation_nll,policy_validation_accuracy,validation_distribution_TV
0,original_linear_64,0.515817,0.343333,1.576064,0.425000,0.159534
1,longer_linear_64,0.515899,0.344583,1.574769,0.426667,0.161069
2,contrast_linear_128,0.537188,0.310000,1.674926,0.370000,0.183049
3,nonlinear_64,0.633531,0.254167,2.974546,0.250000,0.398456
4,nonlinear_contrast_128,0.636279,0.290833,2.543109,0.398333,0.285122
5,prior_linear_l2_003,0.558670,0.354583,1.539228,0.480000,0.128385
6,prior_linear_l2_01,0.554192,0.359583,1.515481,0.488333,0.127549


Selected: prior_linear_l2_01 (lowest validation NLL)
Pooled-scale population baseline: accuracy 36.6%, NLL 1.5127


## 4. Save the selected model
Keep the original model for comparison. Save selected weights, development vectors and validation probabilities with explicit IDs. Checkpoint reload must reproduce predictions. No final personas or final-test metrics are produced here.


In [5]:
state = states[selected["name"]]
checkpoint_path = OUT / "selected_response_model.npz"
pca_file = "outputs/question_answer_pca.npz" if selected["spec"]["features"] in {"original_64", "prior_65"} else "outputs/question_answer_pca_contrast_128.npz"
features_file = {"original_64": "outputs/question_answer_features_64.npy", "contrast_128": "outputs/question_answer_features_contrast_128.npy", "prior_65": "outputs/question_answer_features_prior_65.npy"}[selected["spec"]["features"]]
np.savez(checkpoint_path, **{"state__" + name: value for name, value in state["weights"].items()},
         architecture=np.array(selected["spec"]["architecture"]), feature_set=np.array(selected["spec"]["features"]),
         training_person_ids=np.array(people["shared_training"]), development_person_ids=np.array(people["development"]),
         development_vectors=state["development_vectors"], scale_priors_file=np.array("outputs/response_scale_priors.json"), regularization=np.array(selected["spec"].get("l2", config["l2_penalty"])), pca_file=np.array(pca_file), features_file=np.array(features_file))
prediction_path = OUT / "development_validation_predictions.npz"
np.savez(prediction_path, probabilities=state["probabilities"], person_ids=np.array(people["development"]),
         item_ids=np.array(val_q), option_counts=np.array([len(items[q]["options"]) for q in val_q]))

features = feature_sets[selected["spec"]["features"]]
with np.load(checkpoint_path, allow_pickle=False) as saved:
    restored = ResponseModel(features.shape[1], str(saved["architecture"])).to(device)
    restored.load_state_dict({name[len("state__"):]: torch.tensor(saved[name], device=device) for name in saved.files if name.startswith("state__")})
    restored.eval(); restored.requires_grad_(False)
    restored_z = torch.tensor(saved["development_vectors"], device=device)
    assert saved["training_person_ids"].tolist() == people["shared_training"]
    assert saved["development_person_ids"].tolist() == people["development"]
val_E, val_mask = question_tensors(features, val_q)
np.testing.assert_allclose(predict(restored, restored_z, val_E, val_mask), state["probabilities"], atol=1e-7, rtol=1e-6)
with np.load(contrast_pca_path, allow_pickle=False) as saved:
    reconstructed = ((centered - saved["mean"]) @ saved["components"].T) / np.sqrt(saved["explained_variance"])
np.testing.assert_allclose(contrast, reconstructed, atol=2e-5, rtol=2e-4)
for path, digest in inputs.items(): assert sha256(ROOT / path) == digest
report = {"status": "passed", "config": config, "input_sha256": inputs, "device": str(device),
          "training_modal_accuracy": training_modal_accuracy, "training_modal_by_question": modal_counts,
          "contrast_pca_variance_retained": float(pca.explained_variance_ratio_.sum()),
          "population_baseline": population_baseline, "candidates": results, "selected": selected["name"],
          "selected_pca_file": pca_file, "selected_features_file": features_file,
          "selected_beats_population_baseline_nll": selected["validation"]["nll"] < population_baseline["nll"],
          "validation_answers_loaded": True, "final_test_answers_loaded": False, "final_personas_constructed": False,
          "limitations": ["Only eight validation questions, including two policy questions; selection may overfit these questions.",
                          "Validation accuracy is not directly comparable with training accuracy on different questions.",
                          "Final test remains sealed; future-policy survey predictions remain unvalidated."],
          "checks": ["only training options fit PCA", "respondent splits disjoint", "five predeclared first-round candidates and two documented adaptive second-round candidates",
                     "development vectors fitted from anchors only", "shared weights unchanged during anchor fitting",
                     "valid normalized probabilities and masked options", "original model metric reproduced",
                     "selected checkpoint round trip", "contrast PCA round trip", "existing inputs unchanged"],
          "files": {str(p.relative_to(ROOT)): {"sha256": sha256(p), "bytes": p.stat().st_size}
                    for p in [contrast_path, contrast_pca_path, prior_feature_path, prior_path, checkpoint_path, prediction_path]}}
(OUT / "model_comparison.json").write_text(json.dumps(report, indent=2) + "\n")
print("Saved comparison, selected checkpoint and development validation predictions. Final test remains sealed.")


Saved comparison, selected checkpoint and development validation predictions. Final test remains sealed.
